# Individual edits

Each dermatologist gets their own codebook. That codebook is scored three ways, in order: reliability (original image and question), text generality (one rephrased question), image generality (one perturbed image).

`dermedit` kernel. Set `OVERWRITE = True` to recompute saved runs.

In [ ]:
import sys
from pathlib import Path

PERTURB = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality")
sys.path.insert(0, str(PERTURB))
from utils.bootstrap import bootstrap
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False

EMBED_MODE = "standalone_dual"
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = PERTURB / "results" / "individual"
print("results:", OUT)

In [ ]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_user
from utils.eval import load_suite, load_vlm, render_table, run_suite, suite_complete

rows = []
for model_short in MODELS:
    cached = {}
    todo = []
    for user in USERS:
        if suite_complete(OUT, user, model_short) and not OVERWRITE:
            print(f"{user} / {model_short}: loaded")
            cached[user] = load_suite(OUT, user, model_short)
        else:
            todo.append(user)
    if todo:
        import torch
        editor = None
        vlm, config = load_vlm(
            model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
            embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
            image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
            scaler_dataset=SCALER_DATASET,
        )
        for user in todo:
            edits, _ = collect_user(
                user, model_short, rationale_mode=RATIONALE_MODE,
                norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
            )
            if not edits:
                print(f"skip {user} / {model_short}")
                continue
            editor, queries = edit(vlm, config, edits)
            cached[user] = run_suite(
                editor, vlm, queries, user=user, model_short=model_short,
                out_dir=OUT, seed=SEED, norm_answer=NORM_ANSWER,
                max_new_tokens=MAX_NEW_TOKENS,
            )
        del vlm, editor
        gc.collect()
        torch.cuda.empty_cache()
    for user in USERS:
        if user in cached:
            rows.append(cached[user])

display(Markdown(render_table(rows)))